In [67]:
import math

In [ ]:
# Polynoial Division, necessary for the addition of points in the Elliptic Curve Group
def poly_div(root, poly):
    seq = []
    while len(root) <= len(poly):
        if poly[0] == 0:
            seq += [0]
            poly = poly[1:]
        else:
            value = poly[0] / root[0]
            seq += [value]
            poly[1] = poly[1] + root[1] * (-value)
            poly = poly[1:]
    return seq


# Class of Elliptic Curves. Takes in value A and B to determine the shape of the curve
class EllipticCurve:

    # Initialize
    def __init__(self, A, B):
        assert 4*(A**3) + 27*(B**2) != 0
        self.A = A
        self.B = B

    # Checks if a point lies on the curve
    def is_in_curve(self, p):
        val = (p.x ** 3) + (self.A * p.x) + self.B
        if val == (p.y ** 2):
            return True
        else:
            return False

    # Adds two points under the operation of the Elliptic Curve Group
    def add(self, p1, p2):
        if p1.x == p2.x and p1.y == p2.y:
            slope = (3*(p1.x ** 2) + self.A) / (2 * p1.y)
            y_int = p1.y - slope * p1.x
            poly = [1, -(slope ** 2), (2 * slope * y_int) + self.A, self.B - (y_int ** 2)]
            root = [1, -p1.x]
            poly_2 = poly_div(root, poly)
            poly_3 = poly_div(root, poly_2)
            new_x = -poly_3[1]
            r = Point(new_x, -(slope * new_x + y_int))
            return r
        elif p1 is Omega:
            return p2
        elif p2 is Omega:
            return p1
        elif p1.x == p2.x and p1.y == - p2.y:
            return Omega
        else:
            slope = (p2.y - p1.y) / (p2.x - p1.x)
            y_int = p1.y - (slope * p1.x)
            poly = [1, -(slope ** 2), (2 * slope * y_int) + self.A, self.B - (y_int ** 2)]
            root_1 = [1, -p1.x]
            root_2 = [1, -p2.x]
            poly_2 = poly_div(root_1, poly)
            poly_3 = poly_div(root_2, poly_2)
            new_x = -poly_3[1]
            r = Point(new_x, -(slope * new_x + y_int))
            return r

    # Returns 2P, where P is a point. Also known as, the sum of a point with itself
    def add_self(self, p):
        slope = (3*(p.x ** 2) + self.A) / (2 * p.y)
        y_int = p.y - slope * p.x
        poly = [1, -(slope ** 2), (2 * slope * y_int) + self.A, self.B - (y_int ** 2)]
        root = [1, -p.x]
        poly_2 = poly_div(root, poly)
        poly_3 = poly_div(root, poly_2)
        new_x = -poly_3[1]
        r = Point(new_x, -(slope * new_x + y_int))
        return r



# Class of Points in the Elliptic Curve
class Point:

    def __init__(self, x, y):
        self.x = x
        self.y = y

    def change_x(self, n):
        self.x = n

    def change_y(self, m):
        self.y = m


# Class for the identity element of the Elliptic Curve Group
class Omega(Point):

    x = None
    y = None



In [39]:
E = EllipticCurve(-15, 18)
p = Point(7, 16)
q = Point(1, 2)
r = E.add_self(p)
print([r.x, r.y])
s = E.add(p, q)
[s.x, s.y]


[3.015625, 0.435546875]


[-2.5555555555555545, 6.296296296296297]

In [ ]:
# Class of Elliptic Curves over a Finite Field. Takes in value A and B to determine the shape of the abstract curve. Also takes in a prime number for the finite field.
class EllipticCurveOverField:

    # Initialize
    def __init__(self, A, B, prime):
        assert 4*(A**3) + 27*(B**2) != 0
        assert prime >= 3
        self.A = A
        self.B = B
        self.prime = prime

    # Checks if a point lies on the curve, particularly using the Legendre symbol
    def is_in_curve(self, p):
        value = ((p.x ** 3) + (self.A * p.x) + self.B) % self.prime
        legendre = (value**((self.prime - 1) / 2)) % self.prime
        if legendre == 1:
            return True
        else:
            return False

    # Adds two points under the operation of the Elliptic Curve Group
    def add(self, p1, p2):
        if p1.x == p2.x and p1.y == p2.y:
            value = 2 * p1.y
            for i in range(self.prime):
                if ((value * i) % self.prime) == 1:
                    mult_inv = i
                    break
            slope = ((3*(p1.x ** 2) + self.A) * mult_inv) % self.prime
            y_int = (p1.y - slope * p1.x) % self.prime
            new_x = ((slope ** 2) - (2 * p1.x)) % self.prime
            new_y = (-((slope * new_x) + y_int)) % self.prime
            r = Point(new_x, new_y)
            return r
        elif p1 is Omega:
            return p2
        elif p2 is Omega:
            return p1
        elif p1.x == p2.x and p1.y == (-p2.y) % self.prime:
            return Omega
        else:
            value = p2.x - p1.x
            for i in range(self.prime):
                if ((value * i) % self.prime) == 1:
                    mult_inv = i
                    break
            slope = ((p2.y - p1.y) * mult_inv) % self.prime
            y_int = (p1.y - slope * p1.x) % self.prime
            new_x = ((slope ** 2) - p1.x - p2.x) % self.prime
            new_y = (-((slope * new_x) + y_int)) % self.prime
            r = Point(new_x, new_y)
            return r

    # Returns 2P, where P is a point. Also known as, the sum of a point with itself
    def add_self(self, p):
        value = 2 * p.y
        for i in range(self.prime):
            if ((value * i) % self.prime) == 1:
                mult_inv = i
                break
        slope = ((3*(p.x ** 2) + self.A) * mult_inv) % self.prime
        y_int = (p.y - slope * p.x) % self.prime
        new_x = ((slope ** 2) - (2 * p.x)) % self.prime
        new_y = (-((slope * new_x) + y_int)) % self.prime
        r = Point(new_x, new_y)
        return r

    # Returns the order of a point, also known as the amount of times a point must be added to itself to result in the identity element 
    def order(self, p):
        r = p
        counter = 1
        while r is not Omega:
            q = E.add(r, p)
            r = q
            counter += 1
        return counter
    
    # Solves the Discrete Logarithm Problem. Returns the number of times a point P must be added to itself to result in a point Q
    def logarithm(self, p, q):
        r = p
        value = E.order(p)
        counter = 1
        while r.x != q.x or r.y != q.y:
            s = E.add(r, p)
            r = s
            counter += 1
            print([r.x, r.y])
            if counter > value:
                return None
        return counter

    

    

In [33]:
E = EllipticCurveOverField(-3, 4, 5)
p = Point(0, 2)
q = Point(1, 8)
E.order(p)

7

In [34]:
E = EllipticCurveOverField(8, 7, 73)
p = Point(32, 53)
E.order(p)

41

In [35]:
E = EllipticCurveOverField(8, 7, 73)
p = Point(32, 53)
q = Point(39, 17)
x = E.logarithm(p, q)
print(x)

[25, 65]
[10, 24]
[35, 26]
[14, 4]
[45, 13]
[37, 41]
[1, 4]
[15, 61]
[29, 10]
[39, 17]
11


In [36]:
E = EllipticCurveOverField(8, 7, 73)
p = Point(32, 53)
r = Point(35, 47)
y = E.logarithm(p, r)
print(y)

[25, 65]
[10, 24]
[35, 26]
[14, 4]
[45, 13]
[37, 41]
[1, 4]
[15, 61]
[29, 10]
[39, 17]
[21, 26]
[58, 69]
[8, 46]
[36, 31]
[17, 47]
[30, 50]
[68, 39]
[27, 14]
[69, 38]
[69, 35]
[27, 59]
[68, 34]
[30, 23]
[17, 26]
[36, 42]
[8, 27]
[58, 4]
[21, 47]
[39, 56]
[29, 63]
[15, 12]
[1, 69]
[37, 32]
[45, 60]
[14, 69]
[35, 47]
37


In [37]:
E = EllipticCurveOverField(8, 7, 73)
p = Point(32, 53)
s = Point(58, 4)
z = E.logarithm(p, s)
print(z)

[25, 65]
[10, 24]
[35, 26]
[14, 4]
[45, 13]
[37, 41]
[1, 4]
[15, 61]
[29, 10]
[39, 17]
[21, 26]
[58, 69]
[8, 46]
[36, 31]
[17, 47]
[30, 50]
[68, 39]
[27, 14]
[69, 38]
[69, 35]
[27, 59]
[68, 34]
[30, 23]
[17, 26]
[36, 42]
[8, 27]
[58, 4]
28
